TIỀN XỬ LÝ DỮ LIỆU

- Đọc file

In [5]:
import pandas as pd

df = pd.read_csv(r"C:\Users\hauth\OneDrive\Tài liệu\GitHub\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\BÀI TẬP NHÓM HẰNG TUẦN\LAB5 BÀI TOÁN PHÂN LỚP (NHÓM)\THUẬT TOÁN PHÂN LỚP\2.3.4 Naive mushroom\data\mushrooms.csv")

print(df.head())
print(df.shape)

  class cap-shape cap-surface cap-color bruises odor gill-attachment  \
0     p         x           s         n       t    p               f   
1     e         x           s         y       t    a               f   
2     e         b           s         w       t    l               f   
3     p         x           y         w       t    p               f   
4     e         x           s         g       f    n               f   

  gill-spacing gill-size gill-color  ... stalk-surface-below-ring  \
0            c         n          k  ...                        s   
1            c         b          k  ...                        s   
2            c         b          n  ...                        s   
3            c         n          n  ...                        s   
4            w         b          k  ...                        s   

  stalk-color-above-ring stalk-color-below-ring veil-type veil-color  \
0                      w                      w         p          w   
1       

- Kiểm tra dữ liệu hợp lệ

In [6]:
# Thống kê giá trị thiếu và giá trị được dùng làm ký hiệu thiếu
so_gia_tri_thieu = df.isna().sum()
so_dau_hoi = df.eq("?").sum()
so_chuoi_rong = df.eq("").sum()

bao_cao_hop_le = pd.DataFrame({
    "Giá trị thiếu (NaN)": so_gia_tri_thieu,
    "Ký hiệu '?'": so_dau_hoi,
    "Chuỗi rỗng": so_chuoi_rong
})

print("Kích thước dữ liệu:", df.shape)
print("\nKiểu dữ liệu:")
print(df.dtypes.value_counts())
print("\nCác cột có giá trị cần kiểm tra:")
display(bao_cao_hop_le[
    bao_cao_hop_le.sum(axis=1) > 0
])

# Hiển thị một số dòng có giá trị thiếu hoặc không hợp lệ
dong_can_kiem_tra = (
    df.isna().any(axis=1)
    | df.eq("?").any(axis=1)
    | df.eq("").any(axis=1)
)

print("Số dòng có giá trị cần kiểm tra:", dong_can_kiem_tra.sum())
display(df.loc[dong_can_kiem_tra].head())

Kích thước dữ liệu: (8124, 23)

Kiểu dữ liệu:
str    23
Name: count, dtype: int64

Các cột có giá trị cần kiểm tra:


,Giá trị thiếu (NaN),Ký hiệu '?',Chuỗi rỗng
stalk-root,0,2480,0


Số dòng có giá trị cần kiểm tra: 2480


,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
3984,e,x,y,b,t,n,f,c,b,e,...,s,e,w,p,w,t,e,w,c,w
4023,p,x,y,e,f,y,f,c,n,b,...,s,w,w,p,w,o,e,w,v,p
4076,e,f,y,u,f,n,f,c,n,h,...,f,w,w,p,w,o,f,h,y,d
4100,p,x,y,e,f,y,f,c,n,b,...,s,p,p,p,w,o,e,w,v,d
4104,p,x,y,n,f,f,f,c,n,b,...,s,p,p,p,w,o,e,w,v,l


- Xử lý dữ liệu trùng

In [7]:
# Kiểm tra và loại bỏ các dòng trùng lặp
so_dong_trung = df.duplicated().sum()
print("Số dòng trùng lặp:", so_dong_trung)

df = df.drop_duplicates().reset_index(drop=True)

print("Kích thước dữ liệu sau khi xử lý:", df.shape)

Số dòng trùng lặp: 0
Kích thước dữ liệu sau khi xử lý: (8124, 23)


- Tách đặc trưng và nhãn

In [8]:

# Tách đặc trưng và nhãn
X = df.drop(columns=["class"])
y = df["class"]

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)
print("\nPhân bố nhãn:")
print(y.value_counts())

Kích thước X: (8124, 22)
Kích thước y: (8124,)

Phân bố nhãn:
class
e    4208
p    3916
Name: count, dtype: int64


- Chia train/test

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Kích thước X_train:", X_train.shape)
print("Kích thước X_test:", X_test.shape)
print("\nPhân bố nhãn trong tập train:")
print(y_train.value_counts())
print("\nPhân bố nhãn trong tập test:")
print(y_test.value_counts())

Kích thước X_train: (6499, 22)
Kích thước X_test: (1625, 22)

Phân bố nhãn trong tập train:
class
e    3366
p    3133
Name: count, dtype: int64

Phân bố nhãn trong tập test:
class
e    842
p    783
Name: count, dtype: int64


- Xử lý dữ liệu thiếu

In [10]:
# Các đặc trưng đều là dữ liệu phân loại
cot_phan_loai = X_train.columns.tolist()

# Tránh thay đổi dữ liệu gốc và xử lý dấu '?' như giá trị thiếu
X_train = X_train.copy().replace("?", pd.NA)
X_test = X_test.copy().replace("?", pd.NA)

# Tính mode trên tập train rồi dùng để điền cho cả hai tập
gia_tri_dien = X_train.mode().iloc[0]
X_train = X_train.fillna(gia_tri_dien)
X_test = X_test.fillna(gia_tri_dien)

print("Số giá trị thiếu còn lại trong X_train:", X_train.isna().sum().sum())
print("Số giá trị thiếu còn lại trong X_test:", X_test.isna().sum().sum())

Số giá trị thiếu còn lại trong X_train: 0
Số giá trị thiếu còn lại trong X_test: 0


- Xử lý ngoại lệ

In [11]:
# Kiểm tra các giá trị hiếm trong tập train; không tự động xóa chúng
nguong_hiem = max(1, int(len(X_train) * 0.01))

for cot in cot_phan_loai:
    tan_suat = X_train[cot].value_counts()
    gia_tri_hiem = tan_suat[tan_suat < nguong_hiem]

    if not gia_tri_hiem.empty:
        print(f"{cot} — giá trị xuất hiện dưới {nguong_hiem} lần:")
        print(gia_tri_hiem.to_dict())

print("Không loại dòng nào: các giá trị hiếm vẫn có thể hợp lệ.")

cap-shape — giá trị xuất hiện dưới 64 lần:
{'s': 27, 'c': 4}
cap-surface — giá trị xuất hiện dưới 64 lần:
{'g': 3}
cap-color — giá trị xuất hiện dưới 64 lần:
{'c': 35, 'r': 15, 'u': 10}
odor — giá trị xuất hiện dưới 64 lần:
{'m': 27}
gill-color — giá trị xuất hiện dưới 64 lần:
{'o': 49, 'r': 21}
stalk-surface-above-ring — giá trị xuất hiện dưới 64 lần:
{'y': 17}
stalk-color-above-ring — giá trị xuất hiện dưới 64 lần:
{'c': 27, 'y': 5}
stalk-color-below-ring — giá trị xuất hiện dưới 64 lần:
{'c': 27, 'y': 19}
veil-color — giá trị xuất hiện dưới 64 lần:
{'y': 5}
ring-number — giá trị xuất hiện dưới 64 lần:
{'n': 27}
ring-type — giá trị xuất hiện dưới 64 lần:
{'f': 38, 'n': 27}
spore-print-color — giá trị xuất hiện dưới 64 lần:
{'r': 61, 'y': 38, 'o': 37, 'b': 36, 'u': 34}
Không loại dòng nào: các giá trị hiếm vẫn có thể hợp lệ.


- Mã hóa dữ liệu phân loại

In [12]:
# Mã hóa one-hot các đặc trưng phân loại
X_train_ma_hoa = pd.get_dummies(X_train, dtype=int)
X_test_ma_hoa = pd.get_dummies(X_test, dtype=int)

# Đảm bảo tập test có đúng các cột và thứ tự như tập train
X_test_ma_hoa = X_test_ma_hoa.reindex(
    columns=X_train_ma_hoa.columns,
    fill_value=0
)

X_train = X_train_ma_hoa
X_test = X_test_ma_hoa

print("Kích thước X_train sau mã hóa:", X_train.shape)
print("Kích thước X_test sau mã hóa:", X_test.shape)
display(X_train.head())

Kích thước X_train sau mã hóa: (6499, 116)
Kích thước X_test sau mã hóa: (1625, 116)


,cap-shape_b,cap-shape_c,cap-shape_f,cap-shape_k,cap-shape_s,cap-shape_x,cap-surface_f,cap-surface_g,cap-surface_s,cap-surface_y,...,population_s,population_v,population_y,habitat_d,habitat_g,habitat_l,habitat_m,habitat_p,habitat_u,habitat_w
5249,0,0,1,0,0,0,0,0,0,1,...,0,0,1,0,1,0,0,0,0,0
5781,0,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
7586,1,0,0,0,0,0,0,0,1,0,...,1,0,0,0,1,0,0,0,0,0
6181,0,0,1,0,0,0,0,0,1,0,...,0,1,0,1,0,0,0,0,0,0
7338,0,0,0,1,0,0,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0


- Xử lý mất cân bằng

In [13]:
# Tăng mẫu các lớp thiểu số đến bằng số mẫu của lớp lớn nhất
ten_nhan = "__nhan__"
du_lieu_train = pd.concat(
    [X_train, y_train.rename(ten_nhan)],
    axis=1
)

so_mau_lon_nhat = y_train.value_counts().max()
cac_nhom_can_bang = []

for _, nhom in du_lieu_train.groupby(ten_nhan):
    cac_nhom_can_bang.append(
        nhom.sample(
            n=so_mau_lon_nhat,
            replace=len(nhom) < so_mau_lon_nhat,
            random_state=42
        )
    )

du_lieu_train_can_bang = (
    pd.concat(cac_nhom_can_bang)
    .sample(frac=1, random_state=42)
)

X_train_can_bang = du_lieu_train_can_bang.drop(columns=ten_nhan)
y_train_can_bang = du_lieu_train_can_bang[ten_nhan].rename(y_train.name)

print("Phân bố nhãn trước khi cân bằng:")
print(y_train.value_counts())
print("\nPhân bố nhãn sau khi cân bằng:")
print(y_train_can_bang.value_counts())
print("\nKích thước X_train_can_bang:", X_train_can_bang.shape)

Phân bố nhãn trước khi cân bằng:
class
e    3366
p    3133
Name: count, dtype: int64

Phân bố nhãn sau khi cân bằng:
class
e    3366
p    3366
Name: count, dtype: int64

Kích thước X_train_can_bang: (6732, 116)


- Lưu dữ liệu

In [14]:
import os

thu_muc_luu = "du_lieu_da_xu_ly"
os.makedirs(thu_muc_luu, exist_ok=True)

# Tập train trước khi cân bằng
train_df = X_train.reset_index(drop=True).copy()
train_df["class"] = y_train.reset_index(drop=True)
train_df.to_csv(
    os.path.join(thu_muc_luu, "train.csv"),
    index=False
)

# Tập train đã cân bằng
train_can_bang_df = X_train_can_bang.reset_index(drop=True).copy()
train_can_bang_df["class"] = y_train_can_bang.to_numpy()
train_can_bang_df.to_csv(
    os.path.join(thu_muc_luu, "train_can_bang.csv"),
    index=False
)

# Tập test
test_df = X_test.reset_index(drop=True).copy()
test_df["class"] = y_test.reset_index(drop=True)
test_df.to_csv(
    os.path.join(thu_muc_luu, "test.csv"),
    index=False
)

print(f"Đã lưu các tệp CSV trong thư mục: {thu_muc_luu}")

Đã lưu các tệp CSV trong thư mục: du_lieu_da_xu_ly
